# CSE 4221: Natural Language Processing

## Assignment 01: IMDb Sentiment Analysis

### Team Members:
- Mohammad Abir Rahman (2007026)
- Md. Mushfiqur Rahman (2007027)
- Md. Mezbaur Rahman (2007028)
- Rahimeen Fatin (2007029)
- Raufun Ahsan (2007030)

### Models and Embeddings used in this assignment:

**Models:** Logistic Regression, SVM, Bi-LSTM, Bi-GRU

**Embeddings:** BOW, TF-IDF, Word2Vec (300d), GloVe (300d)

In [14]:
%pip install datasets pandas scikit-learn tensorflow matplotlib gensim -q

In [15]:
import os
import joblib
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GRU, Dense
import gensim.downloader as api

import warnings
warnings.filterwarnings('ignore')

results_db = []
save_dir = 'models'
os.makedirs(save_dir, exist_ok=True)

# 1. Download and prepare the IMDb dataset (50K reviews, 80:20 train/test split).

In [16]:
folder = 'dataset'
os.makedirs(folder, exist_ok=True)
imdb_file_path = os.path.join(folder, 'IMDB_Dataset.csv')

if not os.path.exists(imdb_file_path):
    print("Downloading IMDb dataset...")
    dataset = load_dataset("imdb")
    df_train = pd.DataFrame(dataset['train'])
    df_test = pd.DataFrame(dataset['test'])
    df_combined = pd.concat([df_train, df_test], ignore_index=True)

    # Map labels: 0=negative, 1=positive
    df_combined['sentiment'] = df_combined['label']
    df_combined.rename(columns={'text': 'review'}, inplace=True)
    df_combined[['review', 'sentiment']].to_csv(imdb_file_path, index=False)
    print("Dataset saved!")

# Load and Split (80:20)
df = pd.read_csv(imdb_file_path)
X = df['review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training size: {len(X_train)}, Testing size: {len(X_test)}")


Training size: 40000, Testing size: 10000


## 2. Machine Learning Models (LR & SVM) with BOW & TF-IDF

In [17]:
print("Extracting features (BOW & TF-IDF)...")
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Saving Vectorizers...")

joblib.dump(bow_vectorizer, os.path.join(save_dir, 'bow_vectorizer.pkl'))
joblib.dump(tfidf_vectorizer, os.path.join(save_dir, 'tfidf_vectorizer.pkl'))

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "SVM (Linear)": LinearSVC(max_iter=1000)
}

features = {"BOW": (X_train_bow, X_test_bow), "TF-IDF": (X_train_tfidf, X_test_tfidf)}

for feature_name, (X_tr, X_te) in features.items():
    for model_name, model in models.items():
        print(f"Training {model_name} with {feature_name}...")
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        results_db.append({'Model': f"{model_name} ({feature_name})", 'Accuracy': acc, 'F1 Score': f1})

        print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}\n")

        # Save
        safe_model_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        safe_feature_name = feature_name.lower().replace("-", "")
        filename = f"{safe_model_name}_{safe_feature_name}.pkl"

        joblib.dump(model, os.path.join(save_dir, filename))
        print(f"Saved {model_name} ({feature_name}) to disk as {filename}\n\n")

Extracting features (BOW & TF-IDF)...
Saving Vectorizers...
Training Logistic Regression with BOW...
Accuracy: 0.8709 | F1: 0.8707

Saved Logistic Regression (BOW) to disk as logistic_regression_bow.pkl
Training SVM (Linear) with BOW...
Accuracy: 0.8615 | F1: 0.8610

Saved SVM (Linear) (BOW) to disk as svm_linear_bow.pkl
Training Logistic Regression with TF-IDF...
Accuracy: 0.8914 | F1: 0.8914

Saved Logistic Regression (TF-IDF) to disk as logistic_regression_tfidf.pkl
Training SVM (Linear) with TF-IDF...
Accuracy: 0.8826 | F1: 0.8827

Saved SVM (Linear) (TF-IDF) to disk as svm_linear_tfidf.pkl


## 3. Deep Learning Setup: Tokenization and 300D Embeddings (Word2Vec & GloVe)

In [18]:
folder_name = 'dataset'
glove_zip = os.path.join(folder_name, 'glove.6B.zip')
glove_txt = os.path.join(folder_name, 'glove.6B.300d.txt')

# 1. Download official GloVe
if not os.path.exists(glove_txt):
    if not os.path.exists(glove_zip):
        print("Downloading Official GloVe 6B...")
        url = "http://nlp.stanford.edu/data/glove.6B.zip"
        urllib.request.urlretrieve(url, glove_zip)
        print("Download complete!")

    print("Extracting GloVe 300d text file...")
    with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
        zip_ref.extract('glove.6B.300d.txt', path=folder_name)
    print("GloVe downloaded and extracted successfully!")

# 2. Tokenize Text
max_words = 10000
max_len = 200

print("\nTokenizing texts...")
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)
# Save tokenizer
joblib.dump(tokenizer, os.path.join(save_dir, 'dl_tokenizer.pkl'))

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)
word_index = tokenizer.word_index

# 3. Load Word2Vec
print("\nLoading Word2Vec 300d...")
w2v_model = api.load('word2vec-google-news-300')

# 4. Load GloVe from Text File
print("Parsing GloVe 300d text file...")
glove_index = {}
with open(glove_txt, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove_index[word] = coefs

# 5. Create Embedding Matrices
w2v_matrix = np.zeros((max_words, 300))
glove_matrix = np.zeros((max_words, 300))

for word, i in word_index.items():
    if i < max_words:
        # Fill W2V Matrix
        if word in w2v_model:
            w2v_matrix[i] = w2v_model[word]
        # Fill GloVe Matrix
        if word in glove_index:
            glove_matrix[i] = glove_index[word]

print("Embedding matrices created successfully!")



Tokenizing texts...

Loading Word2Vec 300d...
Parsing GloVe 300d text file...
Embedding matrices created successfully!


## 4. Deep Learning Models (Bi-LSTM & Bi-GRU)

In [ ]:
def build_dl_model(rnn_layer, embedding_matrix):
    model = Sequential([
        Embedding(max_words, 300, weights=[embedding_matrix], input_length=max_len, trainable=False),
        Bidirectional(rnn_layer),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

dl_configs = [
    ("Bi-LSTM (Word2Vec)", LSTM(64), w2v_matrix),
    ("Bi-LSTM (GloVe)", LSTM(64), glove_matrix),
    ("Bi-GRU (Word2Vec)", GRU(64), w2v_matrix),
    ("Bi-GRU (GloVe)", GRU(64), glove_matrix)
]

save_dir = 'models'
os.makedirs(save_dir, exist_ok=True)

for name, layer, matrix in dl_configs:
    print(f"\n--- Training {name} ---")
    model = build_dl_model(layer, matrix)
    # Using 3 epochs
    model.fit(X_train_seq, y_train, epochs=3, batch_size=256, validation_data=(X_test_seq, y_test), verbose=1)

    # Evaluate and store metrics
    preds_probs = model.predict(X_test_seq, verbose=0)
    preds = (preds_probs > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results_db.append({'Model': name, 'Accuracy': acc, 'F1 Score': f1})
    print(f"Result for {name} -> Accuracy: {acc:.4f} | F1: {f1:.4f}")

    name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    filename = f"{name}.keras"

    model.save(os.path.join(save_dir, filename))
    print(f"Saved {name} to disk as {filename}")



--- Training Bi-LSTM (Word2Vec) ---
Epoch 1/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 324s 2s/step - accuracy: 0.7633 - loss: 0.4840 - val_accuracy: 0.7847 - val_loss: 0.4602
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 305s 2s/step - accuracy: 0.8447 - loss: 0.3569 - val_accuracy: 0.8473 - val_loss: 0.3597
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 298s 2s/step - accuracy: 0.8535 - loss: 0.3355 - val_accuracy: 0.8530 - val_loss: 0.3392
Result for Bi-LSTM (Word2Vec) -> Accuracy: 0.8530 | F1: 0.8407
Saved bi-lstm_word2vec to disk as bi-lstm_word2vec.keras

--- Training Bi-LSTM (GloVe) ---
Epoch 1/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 297s 2s/step - accuracy: 0.7237 - loss: 0.5363 - val_accuracy: 0.7667 - val_loss: 0.4841
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 290s 2s/step - accuracy: 0.8265 - loss: 0.3867 - val_accuracy: 0.8499 - val_loss: 0.3549
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 324s 2s/step - accuracy: 0.8552 - loss: 0.3348 - val_accuracy: 0.8646 - val_loss: 0.3267
Result for Bi-LSTM (GloVe) -> Accuracy: 0

## 6. Result Visualization
Comparing the performance of all 8 configurations.

In [ ]:
results_df = pd.DataFrame(results_db)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df['Model']))
width = 0.35

rects1 = ax.bar(x - width/2, results_df['Accuracy'], width, label='Accuracy', color='#4C72B0')
rects2 = ax.bar(x + width/2, results_df['F1 Score'], width, label='F1 Score', color='#DD8452')

ax.set_ylabel('Scores')
ax.set_title('Model Performance Comparison: IMDb Sentiment Analysis', pad=20, fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.set_ylim([0.75, 0.95])
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

display(results_df)

# 7. Load Pre-trained Models from File

In [ ]:
import os
import joblib
from tensorflow.keras.models import load_model

save_dir = 'models'
max_len = 200

print("Loading Processors...")
bow_vectorizer = joblib.load(os.path.join(save_dir, 'bow_vectorizer.pkl'))
tfidf_vectorizer = joblib.load(os.path.join(save_dir, 'tfidf_vectorizer.pkl'))
tokenizer = joblib.load(os.path.join(save_dir, 'dl_tokenizer.pkl'))

print("Loading Machine Learning Models...")
ml_models = {
    "Logistic Regression (BOW)": joblib.load(os.path.join(save_dir, 'logistic_regression_bow.pkl')),
    "Logistic Regression (TF-IDF)": joblib.load(os.path.join(save_dir, 'logistic_regression_tfidf.pkl')),
    "SVM Linear (BOW)": joblib.load(os.path.join(save_dir, 'svm_linear_bow.pkl')),
    "SVM Linear (TF-IDF)": joblib.load(os.path.join(save_dir, 'svm_linear_tfidf.pkl'))
}

print("Loading Deep Learning Models...")
dl_models = {
    "Bi-LSTM (Word2Vec)": load_model(os.path.join(save_dir, 'bi-lstm_word2vec.keras')),
    "Bi-LSTM (GloVe)": load_model(os.path.join(save_dir, 'bi-lstm_glove.keras')),
    "Bi-GRU (Word2Vec)": load_model(os.path.join(save_dir, 'bi-gru_word2vec.keras')),
    "Bi-GRU (GloVe)": load_model(os.path.join(save_dir, 'bi-gru_glove.keras'))
}

print("\nAll 8 models and processors successfully restored! You can now run the inference cell.")

# 8. Infer from New Input


In [ ]:
import pandas as pd
from IPython.display import display
from tensorflow.keras.preprocessing.sequence import pad_sequences

input = input("Enter a movie review to test: ")

if input.strip() == "":
    print("Please enter a valid review.")
else:
    seq_bow = bow_vectorizer.transform([input])
    seq_tfidf = tfidf_vectorizer.transform([input])
    seq_dl = pad_sequences(tokenizer.texts_to_sequences([input]), maxlen=max_len)

    results = []

    def get_label(prediction):
        return "Positive" if prediction == 1 else "Negative"

    for name, model in ml_models.items():
        input_data = seq_bow if "BOW" in name else seq_tfidf
        pred = model.predict(input_data)[0]

        results.append({
            "Model Architecture": name,
            "Sentiment Prediction": get_label(pred),
            "Confidence Score": "N/A (Standard ML Predict)"
        })

    for name, model in dl_models.items():
        prob = model.predict(seq_dl, verbose=0)[0][0]
        pred = 1 if prob > 0.5 else 0

        results.append({
            "Model Architecture": name,
            "Sentiment Prediction": get_label(pred),
            "Confidence Score": f"{prob:.1%} Likelihood of Positive"
        })

    print(f"\nReview Analysis:\n\"{input}\"\n")
    df_results = pd.DataFrame(results)

    display(df_results)